# !!!! Adjust core processing windowns classification!!!!

In [1]:
import os
import time
from pathlib import Path
from datetime import timedelta

import laspy

import preprocessing as pre
import debug_processing as pro

import config.config as configuration

In [21]:
config = configuration.Configuration()
# --- EDIT THESE TO YOUR PATHS ---
config.run_name = "test_peel_2025"

config.target_area_dir = "/Users/tillweiss/Desktop/master/code/Lidar/data/area"
config.las_files_dir = "/Users/tillweiss/Desktop/master/code/Lidar/data/lidar"
config.las_footprints_dir = "/Users/tillweiss/Desktop/master/code/Lidar/las_footprints"

config.preprocessed_dir = "/Users/tillweiss/Desktop/master/code/Lidar/preprocessed"
config.results_dir = "/Users/tillweiss/Desktop/master/code/Lidar/results"

# --- optional: first test run settings ---
config.num_workers = 2        # hält RAM & CPU ruhig
config.chunk_size = 300_000   # groß genug, um Overhead zu vermeiden
config.overlap = 0.05         # 5% reicht für Tests
config.resolution = 2.0 

config.create_DSM = True
config.create_DEM = True
config.create_CHM = False

# validate paths + create output dirs
#config.validate()

In [22]:
def count_points(las_path: str) -> int:
    with laspy.open(las_path) as f:
        return int(f.header.point_count)

print("\n========== STEP A: Footprint matching (for logging) ==========")
t0 = time.time()

run_out = os.path.join(config.preprocessed_dir, config.run_name)
os.makedirs(run_out, exist_ok=True)

las_dict = pre.match_footprints(
    target_footprint_dir=config.target_area_dir,
    las_footprint_dir=config.las_footprints_dir,
    las_file_dir=config.las_files_dir,
    out_dir=os.path.join(config.preprocessed_dir, config.run_name),
    threshold=config.overlap,
    filter_date=config.filter_date,
    start_date=config.start_date,
    end_date=config.end_date
)

print(f"Footprint matching finished in {time.time() - t0:.1f}s")

print("\nTiles per target + raw point counts:")
raw_points_by_target = {}
for target, tiles in las_dict.items():
    raw_points = 0
    for tile in tiles:
        raw_points += count_points(tile)

    raw_points_by_target[target] = raw_points
    print(f"  {target}: tiles={len(tiles)} | raw_points={raw_points:,}")



========== STEP A: Footprint matching (for logging) ==========

Matching Lidar footprints...


Checking LAS footprints: 100%|██████████| 2/2 [00:00<00:00, 29.75footprints/s]

Target area: strip_overlap_core matched 0 footprints – falling back to ALL LAS/LAZ in /Users/tillweiss/Desktop/master/code/Lidar/data/lidar



Finding target areas: 100%|██████████| 1/1 [00:00<00:00,  2.77areas/s]

Target area: strip_overlap_core, LAS files found: 2
Footprint matching completed in 0:00:00. Found 1 target areas.
Footprint matching finished in 0.4s

Tiles per target + raw point counts:
  strip_overlap_core: tiles=2 | raw_points=40,437,900


In [23]:
print("\n========== STEP B: preprocess_all(config) ==========")
t1 = time.time()
pre.preprocess_all(config)
print(f"Preprocessing completed in {timedelta(seconds=int(time.time() - t1))}")

# Check cleaned outputs (raw → cleaned)
print("\nCleaned point counts (raw → cleaned):")
run_pre_dir = Path(config.preprocessed_dir) / config.run_name

for target, raw_points in raw_points_by_target.items():
    # expected output naming pattern from your earlier code:
    cleaned_las = run_pre_dir / f"{Path(target).stem}.las"
    if not cleaned_las.exists():
        print(f"  {Path(target).stem}: NOT FOUND -> {cleaned_las}")
        continue

    cleaned_points = count_points(str(cleaned_las))
    removed_pct = 100 * (1 - cleaned_points / raw_points) if raw_points else 0.0
    print(f"  {Path(target).stem}: {raw_points:,} → {cleaned_points:,} ({removed_pct:.1f}% removed)")



========== STEP B: preprocess_all(config) ==========

========== Starting Preprocessing ==========

--- Matching footprints to LAS files ---

Matching Lidar footprints...


Checking LAS footprints: 100%|██████████| 2/2 [00:00<00:00, 33.37footprints/s]

Target area: strip_overlap_core matched 0 footprints – falling back to ALL LAS/LAZ in /Users/tillweiss/Desktop/master/code/Lidar/data/lidar



Finding target areas: 100%|██████████| 1/1 [00:00<00:00,  3.76areas/s]


Target area: strip_overlap_core, LAS files found: 2
Footprint matching completed in 0:00:00. Found 1 target areas.

--- Merging and Cleaning LAS files ---

Processing LAS files in chunks...


Processing target areas: 100%|██████████| 1/1 [00:00<00:00, 951.52area/s]


Skipping strip_overlap_core: Already processed.

Processing completed in 0:00:00.

Preprocessing completed in 0:00:00.

Preprocessing completed in 0:00:00

Cleaned point counts (raw → cleaned):
  strip_overlap_core: 40,437,900 → 0 (100.0% removed)


In [20]:
print("\n========== STEP C: process_all(config) ==========")
t2 = time.time()
pro.process_all(config)
print(f"Processing completed in {timedelta(seconds=int(time.time() - t2))}")

print("\nDone.")
print(f"Cleaned LAS: {Path(config.preprocessed_dir) / config.run_name}")
print(f"Results:     {Path(config.results_dir) / config.run_name}")


========== STEP C: process_all(config) ==========
Starting Processing ...
[DEBUG] flags: DSM True DEM(flag actually runs DTM) True CHM False

========== Starting DSM Generation ==========
[DEBUG] DSM input folder: /Users/tillweiss/Desktop/master/code/Lidar/preprocessed/test_peel_2025
[DEBUG] DSM found LAS/LAZ: ['/Users/tillweiss/Desktop/master/code/Lidar/preprocessed/test_peel_2025/strip_overlap_core.las']


Processing LAS files: 100%|██████████| 1/1 [00:00<00:00, 136.84file/s]



[DEBUG] saving temp files to /Users/tillweiss/Desktop/master/code/Lidar/results/test_peel_2025/DSM/temp/strip_overlap_core
[DEBUG] strip_overlap_core: n_chunk_tasks = 0
[DEBUG] No chunk tasks created for strip_overlap_core. Check create_chunks_from_wkt().
[DEBUG] Files under /Users/tillweiss/Desktop/master/code/Lidar/results/test_peel_2025/DSM/temp/strip_overlap_core: 0

DSM generation completed in 0:00:00.

========== Starting DTM Generation (called via create_DEM) ==========
[DEBUG] DTM found LAS/LAZ: ['/Users/tillweiss/Desktop/master/code/Lidar/preprocessed/test_peel_2025/strip_overlap_core.las']


Processing LAS files:   0%|          | 0/1 [00:00<?, ?file/s]

[DEBUG] strip_overlap_core: DTM n_chunk_tasks = 0


Processing DTM Chunks: 0it [00:00, ?it/s]
Processing LAS files: 100%|██████████| 1/1 [00:00<00:00, 18.20file/s]

[DEBUG] Found DTM chunk tif files: 0
No DTM chunks found for strip_overlap_core. Skipping.
[DEBUG] Files under /Users/tillweiss/Desktop/master/code/Lidar/results/test_peel_2025/DTM/temp/strip_overlap_core: 0

DTM generation completed in 0:00:00.

Processing completed in 0:00:00
Processing completed in 0:00:00

Done.
Cleaned LAS: /Users/tillweiss/Desktop/master/code/Lidar/preprocessed/test_peel_2025
Results:     /Users/tillweiss/Desktop/master/code/Lidar/results/test_peel_2025
